In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

# Deep Neural Networks 
## Session 20a - Tensorflow

## Regression using RNN LSTM
<img src='../../prasami_images/prasami_color_tutorials_small.png' style = 'width:400px;' alt="By Pramod Sharma : pramod.sharma@prasami.com"/>

### Data Location 
http://archive.ics.uci.edu/ml/datasets/Individual+household+electric+power+consumption

**Attribute Information:**

1. date: Date in format dd/mm/yyyy 
2. time: time in format hh:mm:ss 
3. global_active_power: household global minute-averaged active power (in kilowatt) 
4. global_reactive_power: household global minute-averaged reactive power (in kilowatt) 
5. voltage: minute-averaged voltage (in volt) 
6. global_intensity: household global minute-averaged current intensity (in ampere) 
7. sub_metering_1: energy sub-metering No. 1 (in watt-hour of active energy). It corresponds to the kitchen, containing mainly a dishwasher, an oven and a microwave (hot plates are not electric but gas powered). 
8. sub_metering_2: energy sub-metering No. 2 (in watt-hour of active energy). It corresponds to the laundry room, containing a washing-machine, a tumble-drier, a refrigerator and a light. 
9. sub_metering_3: energy sub-metering No. 3 (in watt-hour of active energy). It corresponds to an electric water-heater and an air-conditioner.


In [ ]:
###-----------------
### Import Libraries
###-----------------

from pathlib import Path
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 


from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error,r2_score

import tensorflow as tf

from utils.helper import fn_plot_tf_hist

In [ ]:
###----------------------
### Some basic parameters
###----------------------

inpDir = Path('..') / '..' / 'input'
outDir = Path('..') / 'output'
modelDir = Path('..') / 'models'
subDir ='housing'

RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE) # Set Random Seed for reproducible results
tf.random.set_seed(RANDOM_STATE) # setting for Tensorflow as well

TEST_SIZE = 0.2
EPOCHS = 100
BATCH_SIZE = 64
PATIENCE  = 10

# Set parameters for decoration of plots
params = {'legend.fontsize': 'medium',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'large',
          'axes.titlesize':'large',
          'xtick.labelsize':'medium',
          'ytick.labelsize':'medium'
         }

CMAP = plt.cm.coolwarm

plt.rcParams.update(params) # update rcParams

## Basic Hygiene

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU') 

if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

# Importing the data and data processing

In [ ]:
data_df = pd.read_csv(inpDir / subDir / 'household_power_consumption.txt',
                 sep=';',  # semicolon separated 
                 na_values=['nan','?']) # how to handle ? and nan string

In [ ]:
data_df.head(10)

In [ ]:
# Combine 'date' and 'time' columns into a single datetime column
data_df['dt'] = pd.to_datetime(data_df['Date'] + ' ' + data_df['Time'], dayfirst=True)

data_df = data_df.set_index('dt', drop=True)
data_df = data_df.drop(['Date', 'Time'], axis=1)
data_df.head()

In [ ]:
data_df.info()

In [ ]:
data_df.shape

In [ ]:
data_df.describe()

In [ ]:
# Check the columns
data_df.columns

In [ ]:
for col in data_df.columns:
       print('In Column: {0}  - Mean: {2}\n\n Values: {1}\n\n'.format(col, data_df[col].unique(), data_df[col].mean()))

##  Clean up  missing values  'nan' with mean

In [ ]:
# filling nan with mean in any columns

data_df = data_df.fillna(data_df.mean())

In [ ]:
data_df.describe().T

## LSTM Data Preparation and feature engineering

This supervised learning problem will be formulated as predicting the `Global_active_power` at the current time (t) given the `Global_active_power` measurement and other features at the prior time step.

In order to reduce the computation time, and also get a quick result to test the model.  One can resample the data over hour (the original data are given in minutes). This will reduce the size of data from 2075259 to 34589 but keep the overall structure of data.   

In [ ]:
## resampling of data over hour
df_resample = data_df.resample('h').agg({
    'Global_active_power': 'mean',  #  'sum'
    'Global_reactive_power': 'mean',
    'Voltage': 'mean',
    'Global_intensity': 'mean',
    'Sub_metering_1': 'sum',  # Sum makes sense for energy consumption
    'Sub_metering_2': 'sum',
    'Sub_metering_3': 'sum'
}) 
df_resample.shape

In [ ]:
df_resample['target'] = df_resample['Global_active_power'].shift(-1)
df_resample.dropna(inplace=True)

In [ ]:
df_resample.tail()

In [ ]:
split = 365*24
train_df = df_resample.iloc[:split]
test_df = df_resample.iloc[split:]
df_resample.shape, train_df.shape, test_df.shape

In [ ]:
y_train = train_df['target'].to_numpy()
train_feat_df = train_df.drop('target', axis = 1)

y_test = test_df['target'].to_numpy()
test_feat_df = test_df.drop('target', axis = 1)

Note: Scale all features in range of [0,1]. 

In [ ]:
# normalize features
scaler = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(train_feat_df)
X_test = scaler.transform(test_feat_df)

In [ ]:
target_scaler = MinMaxScaler(feature_range=(0, 1))
y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = target_scaler.transform(y_test.reshape(-1, 1))

In [ ]:
X_test.shape, y_test.shape

Need to reshape the inputs into the 3D format as expected by LSTMs, namely [samples, timesteps, features].

In [ ]:
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape) 


# Model Architecture

1. LSTM layer
2. dropout layer
4. Dense layer layer
5. output Layer
6. The input shape will be 1 time step with 7 features.
7. Use the Mean Absolute Error (MAE) loss function and the Adam optimizer.
8. The model will be fit for a number of training epochs with a batch size.

In [ ]:
dor = 0.2
model = tf.keras.Sequential()

initializer = tf.keras.initializers.HeNormal()

model.add(tf.keras.layers.Input(shape=(X_train.shape[1],
                                            X_train.shape[2])))
model.add(tf.keras.layers.LSTM(256,
                               activation='tanh',
                               return_sequences=True,
                               kernel_initializer=initializer))

model.add(tf.keras.layers.Dropout(dor))

model.add(tf.keras.layers.LSTM(128, 
                               activation='tanh',
                               kernel_initializer=initializer))

model.add(tf.keras.layers.Dense(64, activation='relu'))

model.add(tf.keras.layers.Dropout(dor))

model.add(tf.keras.layers.Dense(64, activation='relu'))

model.add(tf.keras.layers.Dropout(dor))

model.add(tf.keras.layers.Dense(1))

loss_fn = tf.keras.losses.MeanSquaredError()

model.compile(loss=loss_fn, 
              optimizer='adam', 
              metrics=[tf.keras.metrics.RootMeanSquaredError()])

> tf.keras.layers.LSTM(
    units, activation='tanh', recurrent_activation='sigmoid', use_bias=True,
    kernel_initializer='glorot_uniform', recurrent_initializer='orthogonal',
    bias_initializer='zeros', unit_forget_bias=True, kernel_regularizer=None,
    recurrent_regularizer=None, bias_regularizer=None, activity_regularizer=None,
    kernel_constraint=None, recurrent_constraint=None, bias_constraint=None,
    dropout=0.0, recurrent_dropout=0.0, implementation=2, return_sequences=False,
    return_state=False, go_backwards=False, stateful=False, time_major=False,
    unroll=False, **kwargs
)

## Arguments: 
Arguments|Description
:--|:--
units|Positive integer, dimensionality of the output space.
activation|Activation function to use. Default: hyperbolic tangent (tanh). If you pass None, no activation is applied (ie. "linear" activation: a(x) = x).
recurrent_activation|Activation function to use for the recurrent step. Default: sigmoid (sigmoid). If you pass None, no activation is applied (ie. "linear" activation: a(x) = x).
use_bias|Boolean (default True), whether the layer uses a bias vector.
kernel_initializer|Initializer for the kernel weights matrix, used for the linear transformation of the inputs. Default: glorot_uniform.
recurrent_initializer|Initializer for the recurrent_kernel weights matrix, used for the linear transformation of the recurrent state. Default: orthogonal.
bias_initializer|Initializer for the bias vector. Default: zeros.
unit_forget_bias|Boolean (default True). If True, add 1 to the bias of the forget gate at initialization. Setting it to true will also force bias_initializer="zeros". This is recommended in Jozefowicz et al..
kernel_regularizer|Regularizer function applied to the kernel weights matrix. Default: None.
recurrent_regularizer|Regularizer function applied to the recurrent_kernel weights matrix. Default: None.
bias_regularizer|Regularizer function applied to the bias vector. Default: None.
activity_regularizer|Regularizer function applied to the output of the layer (its "activation"). Default: None.
kernel_constraint|Constraint function applied to the kernel weights matrix. Default: None.
recurrent_constraint|Constraint function applied to the recurrent_kernel weights matrix. Default: None.
bias_constraint|Constraint function applied to the bias vector. Default: None.
dropout|Float between 0 and 1. Fraction of the units to drop for the linear transformation of the inputs. Default: 0.
recurrent_dropout|Float between 0 and 1. Fraction of the units to drop for the linear transformation of the recurrent state. Default: 0.
implementation|Implementation mode, either 1 or 2. Mode 1 will structure its operations as a larger number of smaller dot products and additions, whereas mode 2 will batch them into fewer, larger operations. These modes will have different performance profiles on different hardware and for different applications. Default: 2.
return_sequences|Boolean. Whether to return the last output. in the output sequence, or the full sequence. Default: False.
return_state|Boolean. Whether to return the last state in addition to the output. Default: False.
go_backwards|Boolean (default False). If True, process the input sequence backwards and return the reversed sequence.
stateful|Boolean (default False). If True, the last state for each sample at index i in a batch will be used as initial state for the sample of index i in the following batch.
time_major|The shape format of the inputs and outputs tensors. If True, the inputs and outputs will be in shape [timesteps, batch, feature], whereas in the False case, it will be [batch, timesteps, feature]. Using time_major = True is a bit more efficient because it avoids transposes at the beginning and end of the RNN calculation. However, most TensorFlow data is batch-major, so by default this function accepts input and emits output in batch-major form.
unroll|Boolean (default False). If True, the network will be unrolled, else a symbolic loop will be used. Unrolling can speed-up a RNN, although it tends to be more memory-intensive. Unrolling is only suitable for short sequences. 

In [ ]:
model.summary()

In [ ]:
checkpoint_filepath = modelDir / subDir / 'lstm.weights.h5'

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor = 'val_loss',
    mode = 'auto',
    save_best_only = True
)


early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=PATIENCE,
    mode='auto',
    baseline =None,
    restore_best_weights=True)


In [ ]:
# fit network
history = model.fit(X_train, y_train,
                    epochs=EPOCHS, 
                    batch_size=BATCH_SIZE,
                    validation_data=(X_test, y_test),
                    verbose=2,
                    shuffle=False,
                    callbacks = [early_stopping_callback, model_checkpoint_callback])

In [ ]:
hist_df = pd.DataFrame(history.history)
hist_df = hist_df.rename({'root_mean_squared_error': 'rmse', 'val_root_mean_squared_error' : 'val_rmse'}, axis=1)


fn_plot_tf_hist(hist_df)

## Making Predictions

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
X_test.shape

In [ ]:
# lets scale the X_test to 2 Dimensional array
X_test = X_test.reshape((X_test.shape[0], 7))
X_test[:2, :]

In [ ]:
# inv scale pred
y_pred_original = target_scaler.inverse_transform(y_pred)
y_test_original = target_scaler.inverse_transform(y_test.reshape(-1, 1))

In [ ]:
# calculate RMSE
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))
print('Test RMSE    : %.3f' % rmse)
r2score = r2_score(y_test_original, y_pred_original)
print('Test R2 Score: %.3f' % r2score)

In [ ]:
res_df = pd.DataFrame({'Global_active_power': y_test_original.flatten(), 
                       'Pred': y_pred_original.flatten(),
                       'Error': y_test_original.flatten() - y_pred_original.flatten()})
res_df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))
ax = axes[0]
res_df.Pred.plot(c = 'DarkGreen', alpha=0.8, ax = ax);
ax.scatter(res_df.index, res_df.Global_active_power, c='DarkBlue', marker ='*');
ax.set_xlabel("Time")
ax.set_ylabel("Global Active Power")
ax.grid()

ax = axes[1]
ax.hist(res_df.Error, bins=50, color='DarkOrange', alpha=0.7);
ax.set_xlabel("Prediction Error (kW)")
ax.set_ylabel("Frequency")
ax.grid()
plt.show()
